# State Migration - 03: Versioned state and migration functions

> **MLCourse - Agentic AI - LangGraph - Module 10**

Notebooks 01 and 02 established the problem and its taxonomy. This one builds
the standard solution, which will look familiar to anyone who has managed a
database schema:

1. **Stamp a version number into the state itself.**
2. **Write one small migration function per version step.**
3. **Run the chain at the graph's entry point**, so any node downstream can
   assume the state is current.

### What you will learn

1. Why the version stamp must live *in the state*, not in your code.
2. Writing a migration **chain** (v1→v2→v3) rather than one big function.
3. Installing it as a **migration node** at the graph entry.
4. Why migrations must be **idempotent**, demonstrated.
5. Retrofitting a version stamp onto checkpoints that never had one - the
   situation you are almost certainly actually in.

Still no LLM calls and no API key.

### Setup


In [ ]:
import os
import sqlite3
from typing import TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph

DB_PATH = "versioned_demo.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

checkpointer = SqliteSaver(sqlite3.connect(DB_PATH, check_same_thread=False))
print("versioned-state demo -- no API key required")


### 1. Put the version in the state

The version stamp has to travel **with the data**, not live in your codebase.

That sounds obvious once said, but the tempting alternative - "we deployed v3
in March, so any thread older than March is v2" - fails immediately. A single
database holds threads written by every version you ever shipped, all mixed
together, and a checkpoint carries no timestamp you can trust for this. The
only reliable way to know what schema a checkpoint follows is to have written
it down *in the checkpoint*.

One integer field is enough.

### The three generations of our state schema


In [ ]:
# A support-ticket agent, evolving over three releases.

CURRENT_SCHEMA_VERSION = 3


class TicketStateV1(TypedDict):
    """Release 1: the original. Note there is no version stamp -- this is the
    realistic starting point, and section 5 deals with it."""
    ticket_id: str
    text: str


class TicketStateV2(TypedDict):
    """Release 2: we added a priority, and (finally) a version stamp."""
    schema_version: int
    ticket_id: str
    text: str
    priority: str


class TicketStateV3(TypedDict):
    """Release 3: 'text' was renamed to 'body' (a breaking rename, per
    notebook 02 Case C), and we now track which team owns the ticket."""
    schema_version: int
    ticket_id: str
    body: str              # renamed from 'text'
    priority: str
    team: str              # new


print(f"CURRENT_SCHEMA_VERSION = {CURRENT_SCHEMA_VERSION}")
print("v1 -> v2: add schema_version, add priority")
print("v2 -> v3: rename text -> body, add team")


### 2. One migration function per step, not one big one

The instinct is to write `migrate_to_current(state)` with a pile of `if`
statements. Resist it. Write **one function per version step**, each of which
only has to know about a single change:

```
v1 --[migrate_v1_to_v2]--> v2 --[migrate_v2_to_v3]--> v3
```

Why the chain is worth the extra structure:

- Each function is small enough to be obviously correct, and to be read by
  someone who was not there when the change was made.
- Adding release 4 means writing **one new function**, not editing a growing
  one and risking the older paths.
- A v1 checkpoint and a v2 checkpoint travel the same code path to reach v3,
  so the rare "very old thread" case is exercised by the same tests as the
  common case - rather than being a separate branch nobody runs.

### The migration chain


In [ ]:
def migrate_v1_to_v2(state: dict) -> dict:
    """v1 -> v2: introduce the version stamp and the priority field.

    Every migration takes a plain dict and returns a plain dict. It does not
    know about graphs, checkpointers, or LangGraph at all -- which is exactly
    what makes it trivial to unit-test.
    """
    return {
        **state,
        "schema_version": 2,
        # The judgement call: what did 'priority' mean before it existed?
        # Tickets created before the concept existed were all handled the
        # same way, so "normal" is the honest translation.
        "priority": state.get("priority", "normal"),
    }


def migrate_v2_to_v3(state: dict) -> dict:
    """v2 -> v3: rename text -> body, and add team ownership."""
    migrated = {
        **state,
        "schema_version": 3,
        # THE RENAME, done properly: carry the old value across rather than
        # letting it be orphaned (notebook 02, Case C).
        "body": state.get("body", state.get("text", "")),
        "team": state.get("team", "unassigned"),
    }
    # Drop the old key so it does not linger and confuse a future reader.
    migrated.pop("text", None)
    return migrated


# The registry: {from_version: function}. Adding release 4 means adding one
# line here and one function above.
MIGRATIONS = {
    1: migrate_v1_to_v2,
    2: migrate_v2_to_v3,
}


def migrate(state: dict) -> dict:
    """Upgrade a state dict to CURRENT_SCHEMA_VERSION, one step at a time."""
    version = state.get("schema_version", 1)     # unstamped == the original
    while version < CURRENT_SCHEMA_VERSION:
        step = MIGRATIONS.get(version)
        if step is None:
            raise RuntimeError(
                f"No migration registered from schema version {version}. "
                "Refusing to guess -- add one to MIGRATIONS."
            )
        state = step(state)
        new_version = state["schema_version"]
        # Guard against a migration that forgets to bump the version, which
        # would otherwise spin here forever.
        if new_version <= version:
            raise RuntimeError(
                f"Migration from v{version} did not advance the version "
                f"(still v{new_version}). This would loop forever."
            )
        version = new_version
    return state


print("migration chain registered:", {k: v.__name__ for k, v in MIGRATIONS.items()})


### Testing the chain in isolation, before any graph is involved


In [ ]:
# Migrations are pure functions on dicts. Test them like pure functions.

v1_state = {"ticket_id": "T-1", "text": "printer is on fire"}
v2_state = {"schema_version": 2, "ticket_id": "T-2",
            "text": "cannot log in", "priority": "high"}
v3_state = {"schema_version": 3, "ticket_id": "T-3", "body": "already current",
            "priority": "low", "team": "billing"}

for label, s in [("v1 (unstamped)", v1_state), ("v2", v2_state), ("v3 (current)", v3_state)]:
    out = migrate(dict(s))
    print(f"{label:<16} -> v{out['schema_version']}  {out}")


Note the v1 case: `text` became `body`, `priority` and `team` got their
defaults, and the version reached 3 - through **two** hops, with neither
function knowing the other existed.

Note also the v3 case: already current, so `migrate` did nothing. That is the
idempotency property, and section 4 explains why it is load-bearing rather
than merely tidy.

### 3. Install the chain as a migration node

Now wire it into the graph. The rule that makes this maintainable:

> **The migration node runs first, and every other node may assume the state
> is at `CURRENT_SCHEMA_VERSION`.**

That single guarantee is what stops migration logic from leaking into your
business nodes. Without it, every node ends up carrying its own defensive
`.get()` calls forever, and you can never tell which defaults are still
needed.

We are going to build this the obvious way first. It will pass its unit tests
and then lose data anyway, for a reason worth meeting head-on.

### Building the versioned graph


In [ ]:
def migration_node(state: TicketStateV3) -> dict:
    """Graph entry point: bring whatever we loaded up to the current schema.

    Returns only the CHANGED keys, which is what LangGraph expects from a
    node -- returning the whole state would also work, but this makes the
    checkpoint diff readable.
    """
    current = dict(state)
    before = current.get("schema_version", 1)
    upgraded = migrate(current)

    if before != upgraded["schema_version"]:
        print(f"    [migration] thread upgraded v{before} -> "
              f"v{upgraded['schema_version']}")

    # Only return keys whose value actually changed.
    return {k: v for k, v in upgraded.items() if current.get(k) != v}


def triage(state: TicketStateV3) -> dict:
    """An ordinary business node. Note the DIRECT [] access -- it is safe
    because the migration node guaranteed these fields exist."""
    urgent = state["priority"] == "high"
    return {"team": "escalations" if urgent else state["team"],
            "body": state["body"].upper()}


builder = StateGraph(TicketStateV3)
builder.add_node("migrate", migration_node)
builder.add_node("triage", triage)
builder.add_edge(START, "migrate")       # <-- migration is always first
builder.add_edge("migrate", "triage")
builder.add_edge("triage", END)

agent_v3 = builder.compile(checkpointer=checkpointer)
print("graph: START -> migrate -> triage -> END")


### Create some legacy threads using the OLD graphs


In [ ]:
# To test migration honestly we need checkpoints genuinely written by the old
# schemas, not v3 states pretending to be old ones.

def old_node_v1(state) -> dict:
    return {"text": state["text"] + " [handled by v1]"}


b1 = StateGraph(TicketStateV1)
b1.add_node("work", old_node_v1)
b1.add_edge(START, "work")
b1.add_edge("work", END)
legacy_v1 = b1.compile(checkpointer=checkpointer)


def old_node_v2(state) -> dict:
    return {"text": state["text"] + " [handled by v2]"}


b2 = StateGraph(TicketStateV2)
b2.add_node("work", old_node_v2)
b2.add_edge(START, "work")
b2.add_edge("work", END)
legacy_v2 = b2.compile(checkpointer=checkpointer)

cfg_old = {"configurable": {"thread_id": "ticket-ancient"}}
cfg_mid = {"configurable": {"thread_id": "ticket-recent"}}

legacy_v1.invoke({"ticket_id": "T-100", "text": "printer on fire"}, cfg_old)
legacy_v2.invoke({"schema_version": 2, "ticket_id": "T-200",
                  "text": "cannot log in", "priority": "high"}, cfg_mid)

print("legacy threads written by the real old graphs:")
print(f"  ticket-ancient (v1): {legacy_v1.get_state(cfg_old).values}")
print(f"  ticket-recent  (v2): {legacy_v2.get_state(cfg_mid).values}")


### Run the v3 agent on both legacy threads


In [ ]:
print("v3 agent picking up the ANCIENT (v1) thread:")
result_old = agent_v3.invoke({"ticket_id": "T-100"}, cfg_old)
print(f"    {result_old}\n")

print("v3 agent picking up the RECENT (v2) thread:")
result_mid = agent_v3.invoke({"ticket_id": "T-200"}, cfg_mid)
print(f"    {result_mid}\n")

print("v3 agent on a BRAND NEW thread:")
cfg_new = {"configurable": {"thread_id": "ticket-new"}}
result_new = agent_v3.invoke(
    {"schema_version": 3, "ticket_id": "T-300", "body": "disk full",
     "priority": "high", "team": "unassigned"}, cfg_new)
print(f"    {result_new}")


### Read that output again - the migration lost the text

The version stamps are right. `priority` and `team` got their defaults. The
brand-new thread is perfect.

And both legacy threads came out with **`body: ''`**.

```
ticket-ancient before:  {'ticket_id': 'T-100', 'text': 'printer on fire [handled by v1]'}
ticket-ancient after :  {'schema_version': 3, ..., 'body': '', ...}
```

The ticket text is gone. This is notebook 02's Case C happening *despite* a
migration function written specifically to prevent it - and the migration
function is not the problem. We unit-tested it a few cells ago and it
handled the rename correctly:

```
v1 (unstamped) -> v3  {..., 'body': 'printer is on fire', ...}
```

The pure function works on a dict. The same function inside a node does not.
That difference is the lesson.

### Why: a node only receives channels its state class declares

`migration_node` never saw `text`, because **`TicketStateV3` does not declare
a `text` channel**.

LangGraph builds the state passed to a node from the channels defined by the
graph's state schema. `text` was removed in v3, so the graph has no such
channel, so the stored value is never loaded - it sits orphaned in the
database exactly as in notebook 02. By the time our migration function runs,
`state.get("text", "")` is asking about a key that was filtered out before the
node was entered.

The migration function was correct. It was simply **handed an already-lossy
input**.

This generalises to a rule that is easy to get wrong:

> **A migration node cannot recover a field the current state schema no longer
> declares.** You cannot migrate data you cannot see.

Let's confirm the data really is still there before fixing it.

### The data is not lost -- the graph just cannot see it


In [ ]:
raw = next(checkpointer.list(cfg_old))
print("what the migration NODE received (v3 channels only):")
print("   text present?  ", "text" in agent_v3.get_state(cfg_old).values)
print()
print("what is actually stored in the raw checkpoint history:")
for t in list(checkpointer.list(cfg_old))[-3:]:
    print(f"   {t.checkpoint['channel_values']}")
print()
print("'text' is still in the database. The v3 graph simply never loads it.")


### The fix: declare the legacy field during the transition

This is precisely the **expand-migrate-contract** pattern notebook 02
recommended, and here is the concrete reason it is not optional ceremony:

1. **Expand** - the new state class declares *both* `text` and `body`, so the
   old channel is still loaded and the migration node can see it.
2. **Migrate** - the migration node copies `text` into `body` (lazily here, or
   in bulk as in notebook 04).
3. **Contract** - once no checkpoint still carries `text`, remove it from the
   state class.

The transitional field is marked `NotRequired`, because new threads will never
have it.

### …and a second filter you have to get past

Widening the *graph's* state class is necessary but **not sufficient**.
LangGraph also narrows each node's input to the channels declared by that
**node's own type annotation**. A node annotated `state: TicketStateV3` will
receive only v3 channels even when the graph's schema is wider.

So the migration node needs the transitional annotation too. Let's prove that
filter exists before relying on it.

### Proof: a node's annotation narrows what it receives


In [ ]:
from typing import NotRequired


class _Narrow(TypedDict):
    a: str


class _Wide(TypedDict):
    a: str
    b: NotRequired[str]


def _node_narrow(state: _Narrow) -> dict:
    print(f"   node annotated _Narrow sees: {sorted(state.keys())}")
    return {}


def _node_wide(state: _Wide) -> dict:
    print(f"   node annotated _Wide   sees: {sorted(state.keys())}")
    return {}


from langgraph.checkpoint.memory import InMemorySaver

print("Graph state class is _Wide in BOTH cases; only the node annotation differs:")
for fn in (_node_narrow, _node_wide):
    gb = StateGraph(_Wide)
    gb.add_node("n", fn)
    gb.add_edge(START, "n")
    gb.add_edge("n", END)
    gb.compile(checkpointer=InMemorySaver()).invoke(
        {"a": "x", "b": "SHOULD BE VISIBLE"},
        {"configurable": {"thread_id": "t"}})

print("\nSo BOTH must include the legacy channel:")
print("  1. the graph's state class  (or it is never loaded from the checkpoint)")
print("  2. the migration node's own annotation (or it is filtered at the node)")


### V3, transitional: legacy channels still declared


In [ ]:
from typing import NotRequired


class TicketStateV3Migrating(TypedDict):
    """The v3 schema DURING the transition (the 'expand' phase).

    Identical to TicketStateV3 except that it still declares the deprecated
    'text' channel, so old checkpoints load it and the migration node can
    move the value across. Delete it in the 'contract' release.
    """
    schema_version: int
    ticket_id: str
    body: str
    priority: str
    team: str
    text: NotRequired[str]      # DEPRECATED: read-only, migration source only


def migration_node_transitional(state: TicketStateV3Migrating) -> dict:
    """Identical body to migration_node -- but annotated with the
    TRANSITIONAL state class, which is what actually makes 'text' visible.
    See the note below: LangGraph narrows a node's input to the channels its
    OWN annotation declares."""
    current = dict(state)
    before = current.get("schema_version", 1)
    upgraded = migrate(current)
    if before != upgraded["schema_version"]:
        print(f"    [migration] thread upgraded v{before} -> "
              f"v{upgraded['schema_version']}")
    return {k: v for k, v in upgraded.items() if current.get(k) != v}


builder2 = StateGraph(TicketStateV3Migrating)
builder2.add_node("migrate", migration_node_transitional)
builder2.add_node("triage", triage)
builder2.add_edge(START, "migrate")
builder2.add_edge("migrate", "triage")
builder2.add_edge("triage", END)

agent_v3_migrating = builder2.compile(checkpointer=checkpointer)

# Fresh legacy threads, since the ones above were already (badly) migrated.
cfg_old2 = {"configurable": {"thread_id": "ticket-ancient-2"}}
cfg_mid2 = {"configurable": {"thread_id": "ticket-recent-2"}}
legacy_v1.invoke({"ticket_id": "T-101", "text": "printer on fire"}, cfg_old2)
legacy_v2.invoke({"schema_version": 2, "ticket_id": "T-201",
                  "text": "cannot log in", "priority": "high"}, cfg_mid2)

print("legacy threads (fresh):")
print(f"  {legacy_v1.get_state(cfg_old2).values}")
print(f"  {legacy_v2.get_state(cfg_mid2).values}\n")

print("v3 (transitional) agent on the ANCIENT thread:")
print(f"    {agent_v3_migrating.invoke({'ticket_id': 'T-101'}, cfg_old2)}\n")
print("v3 (transitional) agent on the RECENT thread:")
print(f"    {agent_v3_migrating.invoke({'ticket_id': 'T-201'}, cfg_mid2)}")


### Now it works

`body` carries the original ticket text on both legacy threads, and `triage`
uppercased it - proof that a real value reached the business node.

Three threads at three schema generations went through the **same graph** and
came out current. The `triage` node used direct `state["body"]` access
throughout and never saw a legacy shape.

Three things had to be true for that, and only the last one is obvious:

1. The **graph's** state class still declared the old channel, so the value
   was loaded from the checkpoint at all.
2. The **migration node's own annotation** still declared it, so it survived
   the per-node input filter.
3. The migration **function** copied it across rather than letting it be
   orphaned.

Miss any one of the three and you get `body: ''` with no error - which is
exactly what the first attempt produced, twice, for two different reasons.

Notice also that `text` is still present in the migrated state. That is the
expand phase doing its job - and it is why the **contract** step matters. Once
a backfill (notebook 04) guarantees no live thread depends on `text`, drop it
from the state class and the orphaned data can be cleaned up.

### 4. Migrations must be idempotent

A migration node runs on **every invocation**, not only the first. A long-lived
thread will pass through it hundreds of times. So `migrate(migrate(x))` must
equal `migrate(x)`.

Ours is idempotent by construction: the `while` loop is driven by
`schema_version`, and a state already at the current version enters no
iterations. But it is worth proving rather than assuming, because the common
way to get this wrong is a migration that *appends* - say, adding an entry to
a history list - which grows without bound on every turn.

### Proving idempotency


In [ ]:
once = migrate({"ticket_id": "T-9", "text": "hello"})
twice = migrate(dict(once))
thrice = migrate(dict(twice))

print(f"after 1 migration: {once}")
print(f"after 2:           {twice}")
print(f"after 3:           {thrice}")
print(f"\nidempotent (1 == 2 == 3): {once == twice == thrice}")

# And through the graph: run the same thread several more times.
print("\nRunning the ancient thread three more times through the v3 graph:")
for i in range(3):
    r = agent_v3_migrating.invoke({"ticket_id": "T-101"}, cfg_old2)
    print(f"  turn {i + 2}: schema_version={r['schema_version']}, "
          f"body={r['body']!r}")
print("\nNo '[migration] upgraded' line printed above -- after the first")
print("upgrade the state is current, so the migration node is a no-op.")


### A non-idempotent migration, for contrast

Here is the mistake, so you can recognise it. It looks entirely reasonable.

### The bug: a migration that is not idempotent


In [ ]:
def bad_migration(state: dict) -> dict:
    """WRONG. Appends to a list every time it runs."""
    audit = state.get("audit", [])
    return {**state, "schema_version": 3, "audit": audit + ["migrated"]}


bad_state = {"schema_version": 2, "ticket_id": "T-BAD"}
for turn in range(1, 5):
    bad_state = bad_migration(bad_state)
    print(f"  turn {turn}: audit = {bad_state['audit']}")

print("\nThe audit list grows on every turn. In a real graph this is a state")
print("channel that gets larger with each invocation until the thread is")
print("unusable -- a slow leak that only shows up on long-lived threads,")
print("which are exactly the ones you cannot easily reset.")

print("\nThe fix: guard on the version, and only transform when below it.")


### 5. Retrofitting a version stamp onto unstamped checkpoints

Here is the awkward reality: **you will start this work with no version stamp
at all.** Nobody adds one in release 1. So how does `migrate()` know that an
unstamped checkpoint is v1?

Our chain uses `state.get("schema_version", 1)` - *absent means version 1*.
That works, and it is the right default, but only because we got the ordering
right: the stamp was introduced in v2, so every unstamped checkpoint is
genuinely v1.

If you introduce stamping later - say at v4, with three unstamped generations
already in the database - a default is not enough. You have to **infer** the
version from the shape of the data.

### Inferring a version from the shape of an unstamped checkpoint


In [ ]:
def infer_version(state: dict) -> int:
    """Best-effort version detection for checkpoints written before stamping.

    Order matters: check for the NEWEST distinguishing field first, so a
    v3-shaped state is not mistaken for v2.
    """
    if "schema_version" in state:
        return int(state["schema_version"])     # stamped: just believe it
    if "body" in state and "team" in state:
        return 3
    if "priority" in state:
        return 2
    return 1


samples = [
    {"ticket_id": "T-a", "text": "old one"},
    {"ticket_id": "T-b", "text": "mid one", "priority": "high"},
    {"ticket_id": "T-c", "body": "new one", "priority": "low", "team": "ops"},
    {"schema_version": 2, "ticket_id": "T-d", "text": "stamped", "priority": "low"},
]

print(f"{'inferred':>9}  state")
print("-" * 72)
for s in samples:
    print(f"{infer_version(s):>9}  {s}")

print("\nThis is heuristic and it is fragile -- it depends on each version")
print("having a distinguishing field. Use it ONCE, as part of a backfill that")
print("writes a real schema_version into every checkpoint, and then delete it.")
print("Do not leave shape-sniffing in your permanent read path.")


### Key takeaways

- The version stamp lives **in the state**, never in your deployment
  timeline. One database holds threads from every release you ever shipped.
- Write **one migration function per version step** and chain them. Adding a
  release means adding one function, not editing a growing one - and old
  checkpoints travel the same tested path as recent ones.
- Migrations are **pure functions on dicts**. Unit-test them without a graph.
- Install the chain as a **migration node at `START`**, so every other node
  can use direct `state[...]` access and stay free of legacy handling.
- **A migration node cannot recover a field it cannot see, and there are two
  separate filters between the checkpoint and your node.** Measured above: a
  correct, unit-tested migration function still produced `body: ''` - first
  because the graph's state class no longer declared `text`, and then again
  because the *node's own annotation* did not. During a rename, add the old
  field as `NotRequired[...]` to **both** - the **expand** phase - and drop it
  only after a backfill. You cannot migrate data you cannot see.
- Migrations must be **idempotent** - the node runs on every turn. A migration
  that appends grows the state without bound on long-lived threads.
- Have `migrate()` **fail loudly** on an unknown version rather than guessing.
- If you add stamping late, **infer** the version from the data's shape once,
  during a backfill, then delete the heuristic.

**One limitation to carry into the next notebook.** Everything here is
*lazy* migration: a thread is upgraded when someone next touches it. A thread
nobody touches keeps its old state indefinitely - and a thread **suspended at
an `interrupt()`** may not be reachable at all, because resuming it runs the
pending node, not the entry point.

**Next:** `04_recovering_old_checkpoints.ipynb` - eager backfill across the
whole checkpoint store, and repairing threads that lazy migration cannot
reach.